# 03: Mechanistic Interpretability - Fine-Tuning
This notebook handles the fine-tuning of `attn-only-2l` on Python code and Prose control datasets.

In [3]:
# === STEP 1: SETUP & CONFIGURATION ===
import os
import sys
import torch
import transformer_lens
from google.colab import userdata

repo_url = 'https://github.com/Mattral/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning/'
repo_name = 'Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning'
repo_path = f'/content/{repo_name}'

if not os.path.exists(repo_path):
    !git clone {repo_url}

os.chdir(repo_path)
if repo_path not in sys.path: sys.path.insert(0, repo_path)

if not hasattr(transformer_lens, '__version__'):
    transformer_lens.__version__ = 'unknown'

from src.model.config import ModelConfig, TrainConfig
from src.model.finetune import run_finetuning

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ModelConfig.hf_subset = 'python-all'
ModelConfig.trust_remote_code = True

model_config = ModelConfig()
model_config.hf_subset = 'python-all'
model_config.trust_remote_code = True

print(f'Ready on device: {device} with subset: {model_config.hf_subset}')

Ready on device: cuda with subset: python-all


In [4]:
# === STEP 2: ENABLE DEBUGGING ===
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
print("CUDA Launch Blocking is now enabled for debugging.")

CUDA Launch Blocking is now enabled for debugging.


## 🐍 Condition 1: Code Fine-Tuning (Primary)

In [ ]:
# === STEP 3: RUN CODE FINE-TUNING ===
# Force the correct subset on the instance to resolve the ValueError
model_config.hf_subset = 'python-all'
model_config.trust_remote_code = True

for seed in [42, 123, 7]:
    print(f'\n=== Starting Code Fine-tuning, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='checkpoints', results_dir='experiments/results')
    # The run_finetuning function will now receive the lowercase 'python-all' subset
    history = run_finetuning(model_config, cfg, run_name=f'code_seed{seed}', device=device, prose_control=False)
    print(f'Completed Code seed {seed}')

## 📖 Condition 2: Prose Fine-Tuning (Control)

In [ ]:
for seed in [42, 123, 7]:
    print(f'\n=== Starting Prose Fine-tuning, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='checkpoints', results_dir='experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'prose_seed{seed}', device=device, prose_control=True)
    print(f'Completed Prose seed {seed}')

### 🛠️ Setup Complete
Repository cloned and dependencies installed. **Please restart the runtime** before proceeding to avoid version conflicts.

# ⚠️ Notebook Cleanup
Old setup and training cells have been removed to ensure the patched configuration is used correctly. Please use the 'Global Configuration' cell above and the 'Training' cells below.

⚠️ **IMPORTANT**: Please go to **Runtime > Restart session** (or `Ctrl+M .`) to ensure all installed packages are correctly loaded before continuing with the cells below.

# Notebook 03: Fine-Tuning Runs

**Goal:** Fine-tune on Python code (primary) and TinyStories prose (mandatory control).

**Outputs:** `checkpoints/code_seed*/`, `checkpoints/prose_seed*/`

**Runtime:** ~80 minutes per condition on Colab T4 GPU.

> **Note:** Both conditions are required. The prose control is mandatory for any domain-shift claim.

## 🚀 Re-Running the Corrected Training
The cells below (Condition 1 and Condition 2) now correctly reference the patched `model_config` with the lowercase `python-all` subset.

### 🔑 How to set up your Hugging Face Token

1.  **Generate a Token**: Go to your [Hugging Face Token Settings](https://huggingface.co/settings/tokens) and create a new token (a 'Read' token is sufficient).
2.  **Add to Colab Secrets**:
    *   Click the **key icon** (🔑) in the left sidebar of Google Colab.
    *   Click **"Add new secret"**.
    *   Set the Name to `HF_TOKEN`.
    *   Paste your token into the Value field.
    *   Toggle the **"Notebook access"** switch to ON.

In [9]:
try:
    from google.colab import userdata
    # This will automatically be used by huggingface_hub if the secret is named HF_TOKEN
    token = userdata.get('HF_TOKEN')
    print("HF_TOKEN successfully loaded from secrets.")
except Exception as e:
    print("HF_TOKEN not found in secrets. Using optional public access.")

HF_TOKEN successfully loaded from secrets.


### 📊 Monitor Training Progress
Use the cell below to check which checkpoints have been saved so far. This is useful for tracking progress during long runs.

In [10]:
import os

def list_checkpoints(base_dir='checkpoints'):
    if not os.path.exists(base_dir):
        print(f"Directory '{base_dir}' does not exist yet.")
        return

    for root, dirs, files in os.walk(base_dir):
        level = root.replace(base_dir, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in sorted(files):
            print(f'{subindent}{f}')

print("Current Checkpoints:")
list_checkpoints()

Current Checkpoints:
checkpoints/
    prose_seed42/
        step_000000.pt
    code_seed42/
        step_000000.pt
